# Hansen Ch.24 Quantile Regression

**Chapter 24 Quantile Regression**

理论推导与**面向初学者的详细注释**见同目录 `Hansen_Ch24_Exercises_Solutions.md`。

> **写给只学过李子奈/陈强的同学：** 分位数回归 = m-估计量用 check 函数 $\rho_\tau(u)=u(\tau-\mathbf{1}\{u<0\})$。
> - **LAD**（$\tau=0.5$）= 中位数回归，对厚尾/异常值**稳健**（线性损失 vs OLS 的二次损失）。
> - 多个 $\tau$ 刻画**全分布**（不只均值）——发现分位数异质效应。
> - 渐近方差 $V=Q_\tau^{-1}\Omega_\tau Q_\tau^{-1}$（夹心再现），$Q_\tau$ 含**条件密度** $f_{e|X}(0)$（sparsity）。

## 理论结论的蒙特卡洛验证（无需外部数据）

验证 Ex 24.4：对称误差下 OLS ≈ LAD（估同一 $\beta$）；厚尾（$t_3$）误差下 LAD 方差更小（更稳健）。

In [ ]:
import numpy as np
from scipy.optimize import minimize_scalar
rng = np.random.default_rng(24)

n = 500; reps = 2000; beta_true = 1.0

for label, err_fn in [
    ("正态 N(0,1)", lambda: rng.standard_normal(n)),
    ("厚尾 t(3)", lambda: rng.standard_t(3, n)),
]:
    ols_est, lad_est = [], []
    for r in range(reps):
        X = rng.standard_normal(n)
        e = err_fn()
        Y = beta_true * X + e
        # OLS
        b_ols = np.sum(X * Y) / np.sum(X ** 2)
        ols_est.append(b_ols)
        # LAD: minimize sum |Y - X*theta| (一元, 用有界优化)
        res = minimize_scalar(lambda th: np.sum(np.abs(Y - X * th)),
                              bounds=(b_ols - 2, b_ols + 2), method='bounded')
        lad_est.append(res.x)
    ols_est = np.array(ols_est)
    lad_est = np.array(lad_est)
    print(f"[{label}]")
    print(f"  OLS: 均值={ols_est.mean():.4f}, 方差={ols_est.var():.6f}")
    print(f"  LAD: 均值={lad_est.mean():.4f}, 方差={lad_est.var():.6f}")
    ratio = lad_est.var() / ols_est.var()
    winner = 'LAD 更优 (厚尾稳健)' if ratio < 1 else 'OLS 更优 (正态高效)'
    print(f"  LAD/OLS 方差比 = {ratio:.4f}  ⇒ {winner}\n")